# TALONIR Tutorial: Building Neural Network Graphs

This tutorial introduces TALONIR, a hardware-agnostic format for representing spiking and hybrid neural networks as directed graphs.

## What You'll Learn

1. What TALONIR is and why it exists
2. Creating nodes (primitives)
3. Building graphs
4. Serialization (HDF5)
5. Validation and inspection
6. CNN and skip connections
7. Ghost and detection primitives

```bash
pip install t1c-talon
```

## 1. Setup and Available Primitives

In [ ]:
import numpy as np
from talon import ir

print(f"TALON IR version: {ir.__version__}")
print(f"\nAvailable primitives: {len(ir.list_primitives())}")
for p in sorted(ir.list_primitives()):
    print(f"  - {p}")

## 2. Creating Nodes

In [ ]:
# Input
input_node = ir.Input(np.array([784]))
print(f"Input shape: {input_node.input_type}")

In [ ]:
# Affine
fc1 = ir.Affine(
    weight=np.random.randn(128, 784).astype(np.float32) * 0.01,
    bias=np.zeros(128, dtype=np.float32),
)
print(f"Affine weight: {fc1.weight.shape}")
print(f"Affine bias: {fc1.bias.shape}")

In [ ]:
# LIF neuron
lif1 = ir.LIF(
    tau=np.ones(128, dtype=np.float32) * 10.0,
    r=np.ones(128, dtype=np.float32),
    v_leak=np.zeros(128, dtype=np.float32),
    v_threshold=np.ones(128, dtype=np.float32),
)
print(f"LIF neurons: {len(lif1.tau)}")

In [ ]:
# IF neuron (no leak)
if_neuron = ir.IF(v_threshold=np.ones(32, dtype=np.float32))
print(f"IF neurons: {len(if_neuron.v_threshold)}")

In [ ]:
# Output layer
fc2 = ir.Affine(
    weight=np.random.randn(10, 128).astype(np.float32) * 0.01,
    bias=np.zeros(10, dtype=np.float32),
)
lif2 = ir.LIF(
    tau=np.ones(10, dtype=np.float32) * 10.0,
    r=np.ones(10, dtype=np.float32),
    v_leak=np.zeros(10, dtype=np.float32),
    v_threshold=np.ones(10, dtype=np.float32),
)
output_node = ir.Output(np.array([10]))
print(f"Output shape: {output_node.output_type}")

## 3. Building a Graph

In [ ]:
nodes = {
    "input": input_node, "fc1": fc1, "lif1": lif1,
    "fc2": fc2, "lif2": lif2, "output": output_node,
}
edges = [
    ("input", "fc1"), ("fc1", "lif1"), ("lif1", "fc2"),
    ("fc2", "lif2"), ("lif2", "output"),
]
graph = ir.Graph(nodes=nodes, edges=edges)
print(f"Nodes: {len(graph.nodes)}, Edges: {len(graph.edges)}, DAG: {graph.is_dag}")

In [ ]:
print("Nodes:")
for name, node in graph.nodes.items():
    print(f"  {name}: {type(node).__name__}")
print("\nEdges:")
for src, dst in graph.edges:
    print(f"  {src} -> {dst}")

## 4. Serialization

In [ ]:
import os
os.makedirs("models", exist_ok=True)
ir.write("models/simple_snn.t1c", graph)
print("Saved")

In [ ]:
loaded = ir.read("models/simple_snn.t1c")
print(f"Loaded: {len(loaded.nodes)} nodes, {len(loaded.edges)} edges")
print(f"Weight diff: {np.abs(fc1.weight - loaded.nodes['fc1'].weight).max():.2e}")

## 5. Cyclic Graphs

In [ ]:
cyc = ir.Graph(nodes=nodes, edges=edges + [("lif2", "fc1")])
print(f"Cyclic: DAG={cyc.is_dag}")

## 6. CNN

In [ ]:
def conv_block(name, ic, oc, ks=3):
    conv = ir.Conv2d(
        weight=np.random.randn(oc, ic, ks, ks).astype(np.float32)*0.1,
        bias=np.zeros(oc, dtype=np.float32),
        stride=(1,1), padding=(ks//2, ks//2),
    )
    lif = ir.LIF(
        tau=np.ones(oc, dtype=np.float32)*10,
        r=np.ones(oc, dtype=np.float32),
        v_leak=np.zeros(oc, dtype=np.float32),
        v_threshold=np.ones(oc, dtype=np.float32),
    )
    return {f"{name}_conv": conv, f"{name}_lif": lif}

cnn = {}
cnn["input"] = ir.Input(np.array([1, 28, 28]))
cnn.update(conv_block("b1", 1, 8))
cnn["pool1"] = ir.MaxPool2d(kernel_size=(2,2), stride=(2,2))
cnn.update(conv_block("b2", 8, 16))
cnn["pool2"] = ir.MaxPool2d(kernel_size=(2,2), stride=(2,2))
cnn["flat"] = ir.Flatten(start_dim=0)
cnn["fc"] = ir.Affine(weight=np.random.randn(10, 16*7*7).astype(np.float32)*0.01, bias=np.zeros(10, dtype=np.float32))
cnn["fc_lif"] = ir.LIF(tau=np.ones(10, dtype=np.float32)*10, r=np.ones(10, dtype=np.float32), v_leak=np.zeros(10, dtype=np.float32), v_threshold=np.ones(10, dtype=np.float32))
cnn["output"] = ir.Output(np.array([10]))
print(f"CNN: {len(cnn)} nodes")
for n, nd in cnn.items():
    print(f"  {n}: {type(nd).__name__}")

## 7. Ghost & Detection Primitives

In [ ]:
# SGhostConv
gc = ir.SGhostConv(
    primary_weight=np.random.randn(16, 8, 1, 1).astype(np.float32)*0.1,
    primary_bias=np.zeros(16, dtype=np.float32),
    cheap_weight=np.random.randn(16, 1, 3, 3).astype(np.float32)*0.1,
    cheap_bias=np.zeros(16, dtype=np.float32),
)
print(f"SGhostConv: primary={gc.primary_weight.shape}, cheap={gc.cheap_weight.shape}")

# NMS
nms = ir.NMS(iou_threshold=0.45, score_threshold=0.25, max_detections=100)
print(f"NMS: iou={nms.iou_threshold}, score={nms.score_threshold}, max={nms.max_detections}")

# SConv (standard spiking conv): out=16, in=8, 3x3 kernel
sc = ir.SConv(weight=np.random.randn(16, 8, 3, 3).astype(np.float32)*0.1, bias=np.zeros(16, dtype=np.float32), stride=(1,1), padding=(1,1))
print(f"SConv: {sc.weight.shape}")

# SDConv (spiking depthwise conv): groups == in_channels == out_channels.
# For 8-channel depthwise, weight shape is (8, 1, 3, 3) with groups=8.
sd = ir.SDConv(weight=np.random.randn(8, 1, 3, 3).astype(np.float32)*0.1, bias=np.zeros(8, dtype=np.float32), stride=(1,1), padding=(1,1), groups=8)
print(f"SDConv (depthwise, 8 ch): {sd.weight.shape}, groups={sd.groups}")

## Summary

36 primitives, graph construction, HDF5 serialization, CNN/residual/ghost architectures.